# ChronoState-KD Lite — Phase 5: Evaluation, serving benchmarks, demo

**Goal:** fill the results table from logs, measure the full serving loop, and produce the demo transcript.

| Section | What | ~Time |
| --- | --- | --- |
| 2 | Quality: base / SFT / KD on GSM8K-100, base / KD on MBPP-50 (teacher numbers from Phase 1) | 15 min |
| 3 | Serving: 6-turn session with 2 jumps — VRAM, TTFT, tok/s; `kv_mode` reuse vs recompute, answer agreement | 3 min |
| 4 | SDPA vs eager decode throughput | 2 min |
| 5 | Demo transcript (the 30-second capture) | 1 min |
| 6 | Final results JSON + markdown table | — |

Runtime → **T4 GPU**. Edit `GITHUB_URL`.

In [ ]:
GITHUB_URL = "https://github.com/<YOUR_GITHUB_USER>/chronostate-kd.git"   # <-- edit
REPO_DIR   = "/content/chronostate-kd"
DRIVE_ROOT = "/content/drive/MyDrive/chronostate"
LOCAL_ADAPTERS = "/content/adapters"
ADAPTERS = {"code": "adapter-code", "reasoning": "adapter-reasoning", "reasoning_sft": "adapter-reasoning-sft"}


## 1. Setup: student + container adapter, stores, jumper, router (embedder on GPU)

In [ ]:
import os, json, time, gc, shutil, statistics, torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))
if not os.path.exists(REPO_DIR):
    !git clone $GITHUB_URL $REPO_DIR
%cd $REPO_DIR
!git pull --ff-only
!pip install -q -r requirements.txt && pip install -q -e .
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(LOCAL_ADAPTERS, exist_ok=True)
for run in ADAPTERS.values():
    if not os.path.exists(f"{LOCAL_ADAPTERS}/{run}"): shutil.copytree(f"{DRIVE_ROOT}/adapters/{run}", f"{LOCAL_ADAPTERS}/{run}")
from chronostate.config import load_config
from chronostate.monitor import Monitor
from chronostate.store import AdapterStore
from chronostate.jump import WeightJumper
from chronostate.router import HybridRouter, make_st_embedder
from chronostate.engine import ChronoStateEngine, session_summary, transcript_markdown
from chronostate.evaluate import eval_gsm8k, eval_mbpp, make_hf_generate
from chronostate.data import SYSTEM_PROMPTS
cfg = load_config("configs/lite.yaml")
mon = Monitor(f"{DRIVE_ROOT}/results/events.jsonl", run="phase5-eval")
os.makedirs("results", exist_ok=True)

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(cfg.models.student)
base = AutoModelForCausalLM.from_pretrained(cfg.models.student, quantization_config=bnb, attn_implementation="sdpa", dtype=torch.float16, device_map={"": 0})
model = PeftModel.from_pretrained(base, f"{LOCAL_ADAPTERS}/{ADAPTERS['code']}", adapter_name="default").eval()
store = AdapterStore().add_all({k: f"{LOCAL_ADAPTERS}/{v}" for k, v in ADAPTERS.items()})
jumper = WeightJumper(model, store)
protos = torch.load("results/prototypes.pt")
embed = make_st_embedder(cfg.models.router_embedder, device="cuda")
router = HybridRouter(embed, protos["prototypes"], beta=cfg.router.beta, stickiness=cfg.markov.stickiness, ema=cfg.markov.ema, cos_temperature=cfg.router.cos_temperature)
print(store, "| VRAM after setup:", f"{torch.cuda.memory_allocated()/1e9:.2f} GB")


## 2. Quality eval
Same harness as the teacher in Phase 1 (greedy, 320 new tokens, batch 8). Variants switch via the jumper; `base` disables the adapter in place.

In [ ]:
teacher = json.load(open("results/teacher_eval.json"))
N_G, N_M = cfg.eval.gsm8k_n, cfg.eval.mbpp_n
gen = make_hf_generate(model, tok, max_new_tokens=320)
quality = {"teacher": {"gsm8k": teacher["gsm8k"]["accuracy"], "mbpp": teacher["mbpp"]["pass_at_1"]}}

def run_variant(name, task):
    out = f"{DRIVE_ROOT}/results/eval_{name}_{task}.jsonl"
    if task == "gsm8k": return eval_gsm8k(gen, n=N_G, batch_size=8, out_jsonl=out, tag=name)["accuracy"]
    return eval_mbpp(gen, n=N_M, batch_size=8, out_jsonl=out, tag=name)["pass_at_1"]

t0 = time.time()
with model.disable_adapter():
    quality["base"] = {"gsm8k": run_variant("base", "gsm8k"), "mbpp": run_variant("base", "mbpp")}
print("base", quality["base"], f"{time.time()-t0:.0f}s")
jumper.jump("reasoning_sft"); quality["sft_reasoning"] = {"gsm8k": run_variant("sft_reasoning", "gsm8k")}; print("sft", quality["sft_reasoning"])
jumper.jump("reasoning");     quality["kd_reasoning"]  = {"gsm8k": run_variant("kd_reasoning", "gsm8k")};  print("kd-reasoning", quality["kd_reasoning"])
jumper.jump("code");          quality["kd_code"]       = {"mbpp":  run_variant("kd_code", "mbpp")};        print("kd-code", quality["kd_code"])
for k, v in quality.items(): mon.log("quality", variant=k, **v)
print(json.dumps(quality, indent=2), f"\ntotal {(time.time()-t0)/60:.1f} min")


### Answer-length / format check on GSM8K
Phase 2 showed the KD adapter writes in the teacher's longer style. Did its answers reach `####` within 320 tokens? (Read from the per-sample logs.)

In [ ]:
from chronostate.monitor import read_events
def fmt_stats(name):
    rows = [json.loads(l) for l in open(f"{DRIVE_ROOT}/results/eval_{name}_gsm8k.jsonl")]
    has = sum('####' in r["raw"] for r in rows); return {"n": len(rows), "frac_with_####": has/len(rows), "acc": sum(r["correct"] for r in rows)/len(rows)}
fmt = {n: fmt_stats(n) for n in ["base", "sft_reasoning", "kd_reasoning", "teacher"]}
print(json.dumps(fmt, indent=1))


## 3. Serving benchmark: 6-turn session with 2 topic switches
Run once with `kv_mode="reuse"` (cache kept across jumps) and once with `"recompute"` (fresh prefill every turn), compare TTFT and whether the answers agree.

In [ ]:
SESSION = [
    "Write a Python function that checks whether a number is prime.",
    "Can you make it faster for large n?",
    "A bakery sells muffins at $2.50 each. If Priya buys 6 muffins and a $4 coffee, how much does she spend?",
    "And if muffins were $3 each instead?",
    "Now write a Python function that computes the total cost given a price and a quantity.",
    "Add a docstring to it.",
]
sessions, summaries = {}, {}
for mode in ["reuse", "recompute"]:
    eng = ChronoStateEngine(model, tok, router, jumper, monitor=mon, kv_mode=mode, max_new_tokens=200)
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    turns = eng.run_session(SESSION)
    sessions[mode] = turns; summaries[mode] = session_summary(turns)
    print(f"--- {mode}: " + json.dumps({k: round(v, 2) for k, v in summaries[mode].items()}))
    for t in turns:
        print(f"  t{t.turn+1} {'JUMP' if t.jumped else '    '} {t.chosen:9s} router {t.router_ms:5.1f} ms  jump {t.jump_ms:4.1f} ms  cache {t.cache_tokens_before:4d}  TTFT {t.ttft_ms:6.0f} ms  {t.gen_tokens:3d} tok @ {t.decode_tok_s:4.1f} tok/s")
# agreement between the two modes (only turns after the first jump can differ)
agree = [sessions["reuse"][i].answer == sessions["recompute"][i].answer for i in range(len(SESSION))]
import difflib
sim = [difflib.SequenceMatcher(None, sessions["reuse"][i].answer, sessions["recompute"][i].answer).ratio() for i in range(len(SESSION))]
print("exact agreement per turn:", agree, " | text similarity:", [round(s, 2) for s in sim])
mon.log("kv_mode_agreement", exact=agree, similarity=sim)


## 4. SDPA vs eager decode throughput (same prompt, 128 tokens, 3 runs each)

In [ ]:
def decode_tps(m, n_new=128, reps=3):
    msgs = [{"role": "system", "content": SYSTEM_PROMPTS["code"]}, {"role": "user", "content": "Explain Python list comprehensions with three examples."}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to("cuda")
    m.generate(**enc, max_new_tokens=8, do_sample=False, pad_token_id=tok.pad_token_id)  # warm-up
    r = []
    for _ in range(reps):
        torch.cuda.synchronize(); t0 = time.perf_counter()
        out = m.generate(**enc, max_new_tokens=n_new, min_new_tokens=n_new, do_sample=False, pad_token_id=tok.pad_token_id)
        torch.cuda.synchronize(); r.append((out.shape[1] - enc["input_ids"].shape[1]) / (time.perf_counter() - t0))
    return statistics.median(r)
attn = {"sdpa": decode_tps(model)}
eager_base = AutoModelForCausalLM.from_pretrained(cfg.models.student, quantization_config=bnb, attn_implementation="eager", dtype=torch.float16, device_map={"": 0})
eager = PeftModel.from_pretrained(eager_base, f"{LOCAL_ADAPTERS}/{ADAPTERS['code']}").eval()
attn["eager"] = decode_tps(eager)
del eager, eager_base; gc.collect(); torch.cuda.empty_cache()
print("decode tok/s:", attn); mon.log("attn_throughput", **attn)


## 5. Demo transcript
The conversation to screen-record: Python question → follow-up → math word problem → follow-up → back to code. Printed with router scores and jump latency inline; also saved as markdown.

In [ ]:
eng = ChronoStateEngine(model, tok, router, jumper, monitor=mon, kv_mode="reuse", max_new_tokens=160)
eng.reset()
for u in SESSION[:5]:
    t = eng.chat(u)
    tag = f"⚡ JUMP → {t.chosen}  ({t.jump_ms:.1f} ms)" if t.jumped else f"   stay   {t.chosen}"
    print(f"\n[{tag}]  sem={t.decision['cos_norm'][t.chosen]:.2f} prior={t.decision['prior'][t.chosen]:.2f}  TTFT {t.ttft_ms:.0f} ms  {t.decode_tok_s:.0f} tok/s")
    print(f"USER: {u}\nASSISTANT: {t.answer[:500]}")
md_txt = transcript_markdown(eng.turns, "ChronoState-KD Lite demo session (T4)")
open("results/demo_transcript.md", "w").write(md_txt)
print("\nlearned T after demo:", json.dumps(router.transition_matrix()))


## 6. Final results table

In [ ]:
p3 = json.load(open("results/phase3_jump_benchmark.json")); p4 = json.load(open("results/phase4_routing.json")); p2 = json.load(open("results/phase2_train_summary.json"))
b1, b7 = p4["betas"]["1.0"], p4["betas"]["0.7"]
final = {
  "gpu": torch.cuda.get_device_name(0), "student": cfg.models.student, "teacher": cfg.models.teacher,
  "quality": quality, "gsm8k_format": fmt,
  "kd_vs_sft_gsm8k_delta_pts": round(100 * (quality["kd_reasoning"]["gsm8k"] - quality["sft_reasoning"]["gsm8k"]), 1),
  "kd_pct_of_teacher": {"gsm8k": round(100 * quality["kd_reasoning"]["gsm8k"] / quality["teacher"]["gsm8k"], 1),
                        "mbpp": round(100 * quality["kd_code"]["mbpp"] / quality["teacher"]["mbpp"], 1)},
  "jump": {"p50_ms": p3["benchmark"]["fp32"]["p50_ms"], "p95_ms": p3["benchmark"]["fp32"]["p95_ms"], "mb": p3["benchmark"]["fp32"]["mb_per_jump"],
           "naive_p50_ms": p3["naive_baseline_ms"]["p50_ms"], "speedup": round(p3["naive_baseline_ms"]["p50_ms"] / p3["benchmark"]["fp32"]["p50_ms"])},
  "routing": {"beta1": b1, "beta0.7": b7},
  "serving": summaries, "kv_mode_agreement": {"exact": agree, "similarity": sim},
  "attn_decode_tok_s": attn,
  "training": {k: {kk: v[kk] for kk in ["steps", "seconds", "final_ce", "final_kl", "adapter_mb", "vram_peak_gb"]} for k, v in p2.items()},
}
json.dump(final, open("results/final_results.json", "w"), indent=2); json.dump(final, open(f"{DRIVE_ROOT}/results/final_results.json", "w"), indent=2)
q = quality; r = summaries["reuse"]; rc = summaries["recompute"]
table = f'''| Metric | Result | Target |
| --- | --- | --- |
| GSM8K-100 exact match: base / SFT / **KD** / teacher | {q["base"]["gsm8k"]:.2f} / {q["sft_reasoning"]["gsm8k"]:.2f} / **{q["kd_reasoning"]["gsm8k"]:.2f}** / {q["teacher"]["gsm8k"]:.2f} | KD ≥ SFT + 2 pts |
| MBPP-50 pass@1: base / **KD** / teacher | {q["base"]["mbpp"]:.2f} / **{q["kd_code"]["mbpp"]:.2f}** / {q["teacher"]["mbpp"]:.2f} | — |
| KD as % of teacher (GSM8K / MBPP) | {final["kd_pct_of_teacher"]["gsm8k"]} % / {final["kd_pct_of_teacher"]["mbpp"]} % | ≥ 85 % |
| Adapter swap p50 / p95 (35 MB, in place) | {final["jump"]["p50_ms"]:.1f} / {final["jump"]["p95_ms"]:.1f} ms | p95 < 45 ms |
| vs naive load_adapter | {final["jump"]["naive_p50_ms"]:.0f} ms → **{final["jump"]["speedup"]}×** | — |
| Routing β=1 → β=0.7: unnecessary jumps / follow-up acc | {b1["unnecessary_jumps"]} → {b7["unnecessary_jumps"]} / {b1["followup_accuracy"]:.2f} → {b7["followup_accuracy"]:.2f} | fewer, ≥ acc |
| Router latency (GPU embedder) | {r["router_ms_mean"]:.1f} ms/turn | — |
| 6-turn session, 2 jumps: peak VRAM | {r["vram_peak_gb"]:.2f} GB | < 3 GB |
| TTFT after a jump: KV reuse vs recompute | {r["ttft_ms_after_jump_mean"]:.0f} vs {rc["ttft_ms_after_jump_mean"]:.0f} ms | — |
| Answer agreement reuse vs recompute (exact / mean sim) | {sum(agree)}/{len(agree)} / {statistics.mean(sim):.2f} | report |
| Decode throughput SDPA / eager | {attn["sdpa"]:.1f} / {attn["eager"]:.1f} tok/s | SDPA ≥ eager |
| Training time per adapter | {min(v["seconds"] for v in p2.values())/60:.0f}–{max(v["seconds"] for v in p2.values())/60:.0f} min | — |'''
open("results/results_table.md", "w").write(table); mon.close(); print(table)


## Gate checklist
- [ ] `results/final_results.json`, `results/results_table.md`, `results/demo_transcript.md` committed
- [ ] every row of the table has a number (including the ones that miss their target)

Report back: the printed table, the `fmt` dict from section 2, the two serving summary lines and the agreement line from section 3.